In [0]:
%run "./Transform"

In [0]:
%run "./Extract"

In [0]:
%run "./Load"

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Apple Analysis Data Engineering Project").getOrCreate()

input_df = (
    spark
    .read.format("csv")
    .option("header",True)
    .load("dbfs:/FileStore/tables/Transaction_Updated.csv")
)

input_df.show()

+--------------+-----------+------------+----------------+
|transaction_id|customer_id|product_name|transaction_date|
+--------------+-----------+------------+----------------+
|            11|        105|      iPhone|      2022-02-01|
|            12|        106|      iPhone|      2022-02-02|
|            13|        107|     AirPods|      2022-02-03|
|            14|        105|     AirPods|      2022-02-04|
|            15|        108|      iPhone|      2022-02-05|
|            16|        106|     MacBook|      2022-02-06|
|            17|        107|      iPhone|      2022-02-07|
|            18|        105|     MacBook|      2022-02-08|
|            19|        108|     AirPods|      2022-02-09|
|            20|        106|     AirPods|      2022-02-10|
+--------------+-----------+------------+----------------+



In [0]:
class FirstWorkFlow:
    """
    ETL Pipeline to generate the data for All customers who bought Airpods after buying iPhone
    """
    def __init__(self):
        pass

    def runner(self):
        
        # Step_1: Extract all Required Data from Different sources
        inputDF = AirpodsAfterIphoneExtractor().extract()

        # Step_2: Implement the transformations of Customers who bought Airpods after buying iPhone
        firstTransformedDF = AirpodsAfterIphoneTransformer().transform(inputDF)

        # Step_3: Load all Required Data from different sinks
        AirpodsAfterIphoneLoader(firstTransformedDF).sink()

# LEAD(product_name) -> Partition By: customer_id and Order By: transaction_date ASC


In [0]:
class SecondWorkFlow:
    """
    ETL Pipeline to generate the data for All customers who bought only Airpods and iPhone
    """
    def __init__(self):
        pass

    def runner(self):
        
        # Step_1: Extract all Required Data from Different sources
        inputDF = AirpodsAfterIphoneExtractor().extract()

        # Step_2: Implement the transformations of Customers who bought Airpods after buying iPhone
        onlyAirpodsAndIphoneDF = OnlyAirpodsAndIphone().transform(inputDF)

        # Step_3: Load all Required Data from different sinks
        OnlyAirpodsAndIphoneLoader(onlyAirpodsAndIphoneDF).sink()

# LEAD(product_name) -> Partition By: customer_id and Order By: transaction_date ASC

In [0]:
class ThirdWorkFlow:
    """
    ETL Pipeline to calculate average time taken by customers to buy AirPods after iPhone
    """
    def __init__(self):
        pass

    def runner(self):
        # Step 1: Extract data
        inputDF = AvgTimeIphoneToAirpodsExtractor().extract()

        # Step 2: Transform data
        transformedDF = AvgTimeIphoneToAirpodsTransformer().transform(inputDF)

        # Step 3: Load data
        AvgTimeIphoneToAirpodsLoader(transformedDF).sink()
        
        return transformedDF

In [0]:
class WorkFlowRunner:
    def __init__(self, name):
        self.name = name

    def runner(self):
        if self.name == "firstWorkFlow":
            return FirstWorkFlow().runner()
        elif self.name == "secondWorkFlow":
            return SecondWorkFlow().runner()
        elif self.name == "thirdWorkFlow":
            return ThirdWorkFlow().runner()
        else:
            raiseValueError(f"Not Implemented for {self.name}")

name = "thirdWorkFlow"

workFlowrunner = WorkFlowRunner(name).runner()